```mermaid
flowchart LR
    A0["00"] --> A1a["01a"] --> A1b["01b"] --> A2["02"] --> A3["03"] --> A4a["04a"] --> A4b["04b"]
    A4b --> A5a["05a"] --> A5b["05b"] --> A6a["06a"] --> A6b["06b"]
    A6b --> A7["07"] --> A8a["08a"] --> A8b["08b"]
    A8b --> A9["09"] --> A10["10"] --> A11["11"] --> A12["12"] 
    
    classDef normal fill:#f8f9fa,stroke:#adb5bd,stroke-width:1px,color:#111;
    classDef done fill:#e8f7f0,stroke:#198754,stroke-width:1.5px,color:#111;
    classDef current fill:#fff3cd,stroke:#ff8c00,stroke-width:2px,color:#111;
    
    class A0,A1a done;
    class A1b current;
    class A2,A3,A4a,A4b,A5a,A5b,A6a,A6b,A7,A8a,A8b,A9,A10,A11,A12 normal;
```

# Corpus Builder: Download Project Gutenberg Philosophy Texts

**Purpose:** Download the selected philosophy texts from Project Gutenberg and prepare them for distribution.

**This is an instructor/technical notebook.** Run this once to build the corpus, then distribute the `corpus/` folder to students.

**What this notebook does:**
1. Loads the selected book IDs from Notebook 01
2. Downloads each text from Project Gutenberg
3. Removes standard PG headers and footers
4. Saves raw and cleaned versions
5. Creates a download log

**Estimated time:** ~20 minutes for 500 books

## Setup

In [ ]:
# -----------------------------
# Import
# -----------------------------

import json
import time
import random
import re
import requests
from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime

print("\n✓ Libraries loaded")

# -----------------------------
# Paths
# -----------------------------
SELECTED_IDS_FILE = './analysis/reports/nb01-selected-ids.json'
OUTPUT_DIR = Path('./data')
RAW_DIR = OUTPUT_DIR / 'raw'
CLEANED_DIR = OUTPUT_DIR / 'processed' / 'cleaned'

# -----------------------------
# Parameters
# -----------------------------
MIN_DELAY = 1.5  # Minimum seconds between requests
MAX_DELAY = 2.5  # Maximum seconds between requests
TIMEOUT = 30     # Request timeout in seconds
RESUME = True    # Skip already downloaded files

print(f"\n✓ Configuration set")
print(f"  Resume mode: {'ON' if RESUME else 'OFF'}")
print('-' * 60)
print(f"  Selected IDs: {SELECTED_IDS_FILE}")
print(f"  Output directory: {OUTPUT_DIR.absolute()}")
print(f"  Raw texts: {RAW_DIR.absolute()}")
print(f"  Cleaned texts: {CLEANED_DIR.absolute()}")

## Part 1: Load Selected Book IDs

In [ ]:
# Load selected book IDs from Notebook 01
with open(SELECTED_IDS_FILE, 'r') as f:
    selected_ids = json.load(f)

print(f"\n✓ Loaded {len(selected_ids)} book IDs")
print(f"\nFirst 10 IDs: {selected_ids[:10]}")
print(f"Last 10 IDs: {selected_ids[-10:]}")

## Part 2: Download Functions

In [ ]:
def get_gutenberg_text_urls(book_id):
    """
    Generate possible URLs for a Project Gutenberg text.
    PG uses different URL patterns so we try multiple formats.
    """
    return [
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",  # UTF-8
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",    # ASCII
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",  # Cache
    ]

def download_text(book_id, timeout=30):
    """
    Download text for a specific book ID.
    Tries multiple URL patterns.
    Returns (success, text_content, url_used, error_message)
    """
    urls = get_gutenberg_text_urls(book_id)
    
    for url in urls:
        try:
            response = requests.get(url, timeout=timeout)
            
            if response.status_code == 200:
                # Try to decode as UTF-8, fallback to latin-1
                try:
                    text = response.content.decode('utf-8')
                except UnicodeDecodeError:
                    text = response.content.decode('latin-1')
                
                return True, text, url, None
            
        except requests.exceptions.Timeout:
            continue  # Try next URL
        except requests.exceptions.RequestException as e:
            continue  # Try next URL
    
    # All URLs failed
    return False, None, None, "All URL patterns failed"

def remove_gutenberg_headers(
    text: str,
    remove_produced_by: bool = True,
    remove_proofreading_credits: bool = True
) -> str:
    """
    Remove Project Gutenberg header and footer, including:
    - START / END markers (supports multi‑line variants and flexible asterisks)
    - "Produced by" preamble (optional)
    - Whole paragraphs containing proofreading credits (optional)

    Parameters
    ----------
    text : str
        The raw text from a Project Gutenberg file.
    remove_produced_by : bool, default True
        If True, remove leading lines starting with "Produced by".
    remove_proofreading_credits : bool, default True
        If True, remove entire paragraphs that contain "Proofreading Team"
        or "This file was produced from images generously made available by".

    Returns
    -------
    str
        Cleaned text with headers, footers, and optional boilerplate removed.
    """
    # 1. Find START marker (flexible asterisks, multi-line support)
    start_patterns = [
        r'\*+\s*START OF TH(?:IS|E) PROJECT GUTENBERG EBOOK[\s\S]+?\*+',
        r'\*+\s*START OF TH(?:IS|E) PROJECT GUTENBERG EBOOK[\s\S]+?\*+',
    ]
    start_pos = 0
    for pattern in start_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            start_pos = match.end()
            break

    # 2. Find END marker (flexible asterisks, multi-line support)
    end_patterns = [
        r'\*+\s*END OF TH(?:IS|E) PROJECT GUTENBERG EBOOK[\s\S]+?\*+',
        r'\*+\s*END OF TH(?:IS|E) PROJECT GUTENBERG EBOOK[\s\S]+?\*+',
    ]
    end_pos = len(text)
    for pattern in end_patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            end_pos = match.start()
            break

    # 3. Extract between markers
    cleaned = text[start_pos:end_pos]

    # 4. (Optional) Remove leading "Produced by" lines
    if remove_produced_by:
        lines = cleaned.splitlines()
        while lines:
            first_line = lines[0]
            if first_line.strip() == '':
                lines.pop(0)
                continue
            if re.match(r'^\s*produced by', first_line, re.IGNORECASE):
                lines.pop(0)
                continue
            break
        cleaned = '\n'.join(lines)

    # 5. (Optional) Remove entire paragraphs containing proofreading credits
    if remove_proofreading_credits:
        paragraphs = re.split(r'\n\s*\n', cleaned)
        filtered_paragraphs = []

        for para in paragraphs:
            if re.search(r'proofreading team', para, re.IGNORECASE):
                continue
            if re.search(r'this file was produced from images generously made available by', para, re.IGNORECASE):
                continue
            if re.search(r'transcribed from', para, re.IGNORECASE):
                continue
            if re.search(r'transcriber', para, re.IGNORECASE):
                continue
            filtered_paragraphs.append(para)

        cleaned = '\n\n'.join(filtered_paragraphs)

    return cleaned.strip()

def compare_pg_ids_with_filenames(
    pg_ids: List[Union[int, str]],
    filenames: List[Union[str, Path]]
) -> dict:
    """
    Compare a list of Project Gutenberg IDs with a list of filenames
    (e.g., 'pg10108.txt') and return missing items in both directions.

    Parameters
    ----------
    pg_ids : list of int or str
        List of Gutenberg book IDs (e.g., [10108, 10112]).
    filenames : list of str or Path
        List of filenames as strings or PosixPath objects.

    Returns
    -------
    dict
        {
            'missing_ids': list,      # IDs not found in filenames
            'missing_filenames': list # Filenames not found in ids
        }
    """
    # Convert all filenames to strings (handle Path objects)
    filenames_str = [str(f) for f in filenames]

    # Normalise pg_ids to strings (remove leading zeros if any)
    id_set = {str(i).lstrip('0') for i in pg_ids}

    # Extract numeric part from filenames (remove 'pg' and '.txt')
    file_id_set = set()
    for f in filenames_str:
        # Extract digits between 'pg' and '.txt'
        match = re.search(r'pg([0-9]+)\.txt', f)
        if match:
            # Remove leading zeros to match integer representation
            file_id_set.add(match.group(1).lstrip('0'))

    # Find missing IDs (in id_set but not in file_id_set)
    missing_ids = sorted(id_set - file_id_set)

    # Find missing filenames (in file_id_set but not in id_set)
    missing_file_ids = sorted(file_id_set - id_set)
    # Convert back to full filenames
    missing_filenames = [f'pg{_id}.txt' for _id in missing_file_ids]

    return {
        'missing_ids': missing_ids,
        'missing_filenames': missing_filenames
    }

print("\n✓ Helper functions defined")

## HTML cleaning functions

In [ ]:
def strip_html_tags(html_text):
    """Basic HTML tag removal."""
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html_text, 'html.parser')
    return soup.get_text()

def clean_html_regex(text: str) -> str:
    # Remove everything between < and >
    text = re.sub(r'<[^>]+>', ' ', text)
    # Remove HTML entities like &nbsp;, &amp;, etc.
    text = re.sub(r'&[a-zA-Z]+;', ' ', text)
    # Remove email addresses (anything containing @)
    text = re.sub(r'\S+@\S+', ' ', text)
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def clean_html_full(text: str) -> str:
    """
    Robust HTML cleaner that:
    - Uses BeautifulSoup to parse and extract text.
    - Removes script and style tags.
    - Replaces HTML entities with spaces.
    - Collapses multiple spaces.
    """
    if not text or not isinstance(text, str):
        return ""

    # 1. Use BeautifulSoup to parse HTML and extract text
    #    Use 'html.parser' or 'lxml' if available (lxml is faster but requires installation)
    soup = BeautifulSoup(text, 'html.parser')
    
    # Remove script and style tags (often contain noise)
    for script in soup(["script", "style"]):
        script.decompose()

    # Get text with spaces between elements, strip extra whitespace
    clean_text = soup.get_text(separator=' ', strip=True)

    # 2. Remove any remaining HTML entities (like &nbsp;, &amp;, etc.)
    clean_text = re.sub(r'&[a-zA-Z]+;', ' ', clean_text)

    # 3. Remove email addresses (if any)
    clean_text = re.sub(r'\S+@\S+', ' ', clean_text)

    # 4. Collapse multiple spaces into one
    clean_text = re.sub(r'\s+', ' ', clean_text)

    return clean_text.strip()

def clean_html_robust(text: str) -> str:
    # First, remove obvious tags with regex (fast)
    text = re.sub(r'<[^>]+>', ' ', text)
    # Then pass to BeautifulSoup for more thorough cleaning
    soup = BeautifulSoup(text, 'html.parser')
    clean_text = soup.get_text(separator=' ', strip=True)
    # Remove entities and collapse spaces
    clean_text = re.sub(r'&[a-zA-Z]+;', ' ', clean_text)
    clean_text = re.sub(r'\s+', ' ', clean_text)
    return clean_text.strip()

## Part 3: Download All Texts

This will download all selected texts from Project Gutenberg.

**Estimated time:** ~20 minutes for 500 books with rate limiting

In [ ]:
# Initialize download log
download_log = {
    'timestamp': datetime.now().isoformat(),
    'total_books': len(selected_ids),
    'successful': [],
    'failed': [],
    'skipped': []  # Already downloaded (if RESUME=True)
}

print("Starting download...")
print(f"Total books to download: {len(selected_ids)}")
print(f"Rate limiting: {MIN_DELAY}-{MAX_DELAY} seconds between requests")
print("\nPress Ctrl+C to interrupt (progress will be saved)\n")

try:
    for book_id in tqdm(selected_ids, desc="Downloading texts"):
        raw_file = RAW_DIR / f'pg{book_id}.txt'
        
        # Check if already downloaded (resume mode)
        if RESUME and raw_file.exists():
            download_log['skipped'].append(book_id)
            continue
        
        # Download
        success, text, url, error = download_text(book_id, timeout=TIMEOUT)
        
        if success:
            # Save raw version
            with open(raw_file, 'w', encoding='utf-8') as f:
                f.write(text)
            
            download_log['successful'].append({
                'book_id': book_id,
                'url': url,
                'size_bytes': len(text)
            })
        else:
            download_log['failed'].append({
                'book_id': book_id,
                'error': error
            })
        
        # Rate limiting with random jitter
        delay = random.uniform(MIN_DELAY, MAX_DELAY)
        time.sleep(delay)

except KeyboardInterrupt:
    print("\n\n⚠️  Download interrupted by user")
    print("Progress has been saved. You can resume later.\n")

# Summary
print("\n" + "=" * 60)
print("DOWNLOAD COMPLETE")
print("=" * 60)
print(f"Successful: {len(download_log['successful'])}")
print(f"Failed: {len(download_log['failed'])}")
print(f"Skipped (already downloaded): {len(download_log['skipped'])}")
if (len(selected_ids)-len(download_log['skipped'])) > 0:
    print(f"\nSuccess rate: {len(download_log['successful'])/(len(selected_ids)-len(download_log['skipped']))*100:.1f}%")

## Part 4: Clean Texts (Remove Headers/Footers)

Remove Project Gutenberg headers and footers from all downloaded texts.

In [ ]:
print("Cleaning texts (removing PG headers/footers, copyrights, html, etc.)...\n")

cleaning_log = {
    'successful': [],
    'failed': []
}

# Get list of raw files to clean
raw_files = list(RAW_DIR.glob('pg*.txt'))

for raw_file in tqdm(raw_files, desc="Cleaning texts"):
    try:
        # Read raw text
        with open(raw_file, 'r', encoding='utf-8') as f:
            raw_text = f.read()
        
        # Remove HTML tags
        text = strip_html_tags(raw_text) # Use differenct functions if HTML tags remain
        # Remove PG headers/footers
        cleaned_text = remove_gutenberg_headers(text)

        # Save cleaned version
        cleaned_file = CLEANED_DIR / raw_file.name
        with open(cleaned_file, 'w', encoding='utf-8') as f:
            f.write(cleaned_text)
        
        # Calculate reduction
        reduction = (len(raw_text) - len(cleaned_text)) / len(raw_text) * 100
        
        cleaning_log['successful'].append({
            'book_id': raw_file.stem.replace('pg', ''),
            'raw_size': len(raw_text),
            'cleaned_size': len(cleaned_text),
            'reduction_pct': reduction
        })
        
    except Exception as e:
        cleaning_log['failed'].append({
            'book_id': raw_file.stem.replace('pg', ''),
            'error': str(e)
        })

print("\n" + "=" * 60)
print("CLEANING COMPLETE")
print("=" * 60)
print(f"Cleaned successfully: {len(cleaning_log['successful'])}")
print(f"Failed: {len(cleaning_log['failed'])}")

if cleaning_log['successful']:
    avg_reduction = sum(item['reduction_pct'] for item in cleaning_log['successful']) / len(cleaning_log['successful'])
    print(f"Average size reduction: {avg_reduction:.1f}%")

# Check the efficiency of `remove_gutenberg_headers` on PG texts

In [ ]:
import difflib
from bs4 import BeautifulSoup

# --- Define paths ---
RAW_DIR = Path('./data/raw')
CLEANED_DIR = Path('./data/processed/cleaned')

# --- Choose a file to test ---
PG_ID = 77427   # change this to test a different book
raw_file = RAW_DIR / f'pg{PG_ID}.txt'
cleaned_file = CLEANED_DIR / f'pg{PG_ID}.txt'

if not raw_file.exists():
    raise FileNotFoundError(f"Raw file not found: {raw_file}")

# --- Read raw text ---
raw_text = raw_file.read_text(encoding='utf-8', errors='replace')

# --- Re‑clean from scratch (to ensure we use the latest functions) ---
cleaned_text = remove_gutenberg_headers(clean_html_robust(raw_text))

# --- Print snippets ---
print("=" * 60)
print(f"RAW FILE: {raw_file.name} (first 500 chars)")
print("=" * 60)
print(raw_text[:500])
print("\n...\n")

print("=" * 60)
print(f"CLEANED (first 500 chars)")
print("=" * 60)
print(cleaned_text[:500])
print("\n...\n")

# --- Statistics ---
print("=" * 60)
print("STATISTICS")
print("=" * 60)
print(f"Raw size: {len(raw_text):,} characters")
print(f"Cleaned size: {len(cleaned_text):,} characters")
print(f"Reduction: {(1 - len(cleaned_text)/len(raw_text))*100:.1f}%")

# --- Optional: compare with existing cleaned file (if any) ---
if cleaned_file.exists():
    existing_cleaned = cleaned_file.read_text(encoding='utf-8', errors='replace')
    print(f"\nExisting cleaned file size: {len(existing_cleaned):,} characters")
    if existing_cleaned == cleaned_text:
        print("✅ The cleaned file matches the re‑cleaned version.")
    else:
        print("⚠️  The cleaned file differs from the re‑cleaned version.")
else:
    print("\nNo existing cleaned file for comparison.")

In [ ]:
def compare_raw_cleaned(
    pg_id: int,
    raw_dir: Path = Path("./data/raw"),
    cleaned_dir: Path = Path("./data/processed/cleaned"),
    preview_chars: int = 500,
    show_diff: bool = True,
    diff_context_lines: int = 3
) -> None:
    """
    Compare a raw Project Gutenberg file with its cleaned version.

    Parameters
    ----------
    pg_id : int
        The Gutenberg book ID (e.g., 340).
    raw_dir : Path
        Directory containing the raw files.
    cleaned_dir : Path
        Directory containing the cleaned files.
    preview_chars : int, default 500
        Number of characters to show from the start of each file.
    show_diff : bool, default True
        If True, display a unified diff between the two texts.
    diff_context_lines : int, default 3
        Number of context lines to show in the diff.
    """
    raw_file = raw_dir / f"pg{pg_id}.txt"
    cleaned_file = cleaned_dir / f"pg{pg_id}.txt"

    if not raw_file.exists():
        raise FileNotFoundError(f"Raw file not found: {raw_file}")
    if not cleaned_file.exists():
        raise FileNotFoundError(f"Cleaned file not found: {cleaned_file}")

    # Read both files
    raw_text = raw_file.read_text(encoding="utf-8", errors="replace")
    cleaned_text = cleaned_file.read_text(encoding="utf-8", errors="replace")

    # --- Statistics ---
    raw_len = len(raw_text)
    cleaned_len = len(cleaned_text)
    reduction = (1 - cleaned_len / raw_len) * 100

    print("=" * 70)
    print(f"COMPARISON: pg{pg_id}.txt")
    print("=" * 70)
    print(f"Raw size:     {raw_len:,} characters")
    print(f"Cleaned size: {cleaned_len:,} characters")
    print(f"Reduction:    {reduction:.1f}%")
    print(f"Files match:  {raw_text == cleaned_text}")
    print()

    # --- Preview ---
    print(f"RAW PREVIEW (first {preview_chars} chars):")
    print("-" * 70)
    print(raw_text[:preview_chars])
    print("\n")

    print(f"CLEANED PREVIEW (first {preview_chars} chars):")
    print("-" * 70)
    print(cleaned_text[:preview_chars])
    print("\n")

    # --- Diff ---
    if show_diff:
        print("UNIFIED DIFF (first few differences):")
        print("-" * 70)
        diff = difflib.unified_diff(
            raw_text.splitlines(),
            cleaned_text.splitlines(),
            fromfile=f"raw/pg{pg_id}.txt",
            tofile=f"cleaned/pg{pg_id}.txt",
            n=diff_context_lines
        )
        # Print only the first 100 lines of the diff to avoid flooding
        diff_lines = list(diff)
        for line in diff_lines[:100]:
            print(line)
        if len(diff_lines) > 100:
            print(f"... (showing first 100 of {len(diff_lines)} diff lines)")

In [ ]:
# Compare pg340.txt (raw vs cleaned)
compare_raw_cleaned(
    pg_id=77427,
    raw_dir=Path("./data/raw"),
    cleaned_dir=Path("./data/processed/cleaned"),
    preview_chars=300,
    show_diff=True
)

# Part 5: Save Download Log

In [ ]:
# Combine download and cleaning logs
full_log = {
    'download': download_log,
    'cleaning': cleaning_log,
    'summary': {
        'total_selected': len(selected_ids),
        'downloaded': len(download_log['successful']),
        'download_failed': len(download_log['failed']),
        'cleaned': len(cleaning_log['successful']),
        'cleaning_failed': len(cleaning_log['failed'])
    }
}

# Save log
log_file = Path('./analysis/reports/nb01-download_log.json')
with open(log_file, 'w', encoding='utf-8') as f:
    json.dump(full_log, f, indent=2, ensure_ascii=False)

print(f"✓ Log saved to {log_file}")

# Display failed downloads if any
if download_log['failed']:
    print("\n⚠️  Failed downloads:")
    for item in download_log['failed'][:10]:  # Show first 10
        print(f"  Book ID {item['book_id']}: {item['error']}")
    if len(download_log['failed']) > 10:
        print(f"  ... and {len(download_log['failed']) - 10} more (see log file)")

if cleaning_log['failed']:
    print("\n⚠️  Cleaning failures:")
    for item in cleaning_log['failed']:
        print(f"  Book ID {item['book_id']}: {item['error']}")

In [ ]:
download_log['failed']

In [ ]:
failed_ids = [item['book_id'] for item in download_log['failed']]

# Second trial

In [ ]:
import requests
import time
import random
from pathlib import Path

# Configuration
TEXTS_RAW_DIR = Path('./data/raw')
TEXTS_CLEANED_DIR = Path('./data/processed/cleaned')
RETRY_IDS = failed_ids  # Edit if necessary

def get_all_text_urls(book_id):
    """
    Generate all possible URLs for a book, including special formats.
    """
    return [
        # Standard text formats
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-h/{book_id}-h.htm",

        f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
        
        # HTML (we can strip tags if needed)
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-h/{book_id}-h.htm",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-h.htm",
        
        # UTF-8 variants
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-8.txt",
        
        # TeX/LaTeX (if you want to try)
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-t/{book_id}-t.tex",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.tex",
    ]

def download_single_book(book_id, timeout=30):
    """
    Try to download a single book, trying all possible URLs.
    Returns (success, text_content, url_used, format_found)
    """
    urls = get_all_text_urls(book_id)
    
    for url in urls:
        try:
            print(f"  Trying: {url}")
            response = requests.get(url, timeout=timeout)
            
            if response.status_code == 200:
                # Decode content
                try:
                    text = response.content.decode('utf-8')
                except UnicodeDecodeError:
                    text = response.content.decode('latin-1')
                
                # Determine format
                if url.endswith('.tex'):
                    format_found = 'tex'
                elif url.endswith('.htm') or url.endswith('.html'):
                    format_found = 'html'
                else:
                    format_found = 'txt'
                
                print(f"  ✓ Success! Format: {format_found}")
                return True, text, url, format_found
                
        except requests.exceptions.Timeout:
            print(f"  ✗ Timeout")
            continue
        except requests.exceptions.RequestException as e:
            print(f"  ✗ Error: {e}")
            continue
    
    print(f"  ✗ All URLs failed")
    return False, None, None, None


# Retry downloading failed books
print(f"Retrying {len(RETRY_IDS)} failed downloads...\n")

results = {
    'success': [],
    'failed': [],
    'special_format': []
}

for book_id in RETRY_IDS:
    print(f"Book {book_id}:")
    
    success, text, url, format_found = download_single_book(book_id)
    
    if success:
        # Save raw
        raw_file = TEXTS_RAW_DIR / f'pg{book_id}.txt'
        with open(raw_file, 'w', encoding='utf-8') as f:
            f.write(text)
        
        # Clean based on format
        if format_found == 'html':
            cleaned_text = strip_html_tags(text)
            cleaned_text = remove_gutenberg_headers(cleaned_text)
            results['special_format'].append((book_id, 'html'))
        elif format_found == 'tex':
            # For TeX, just remove headers but keep LaTeX commands
            cleaned_text = remove_gutenberg_headers(text)
            results['special_format'].append((book_id, 'tex'))
            print(f"  ⚠️  TeX format - may need manual review")
        else:
            cleaned_text = remove_gutenberg_headers(text)
        
        # Save cleaned
        cleaned_file = TEXTS_CLEANED_DIR / f'pg{book_id}.txt'
        with open(cleaned_file, 'w', encoding='utf-8') as f:
            f.write(cleaned_text)
        
        results['success'].append(book_id)
        print(f"  ✓ Saved to {raw_file.name}\n")
        
    else:
        results['failed'].append(book_id)
        print(f"  ✗ Could not download\n")
    
    # Be respectful
    time.sleep(random.uniform(1.5, 2.5))

# Report
print("\n" + "=" * 60)
print("RETRY RESULTS")
print("=" * 60)
print(f"Successful downloads: {len(results['success'])}")
print(f"  IDs: {results['success']}")

if results['special_format']:
    print(f"\nSpecial formats found: {len(results['special_format'])}")
    for book_id, fmt in results['special_format']:
        print(f"  Book {book_id}: {fmt}")

print(f"\nStill failed: {len(results['failed'])}")
if results['failed']:
    print(f"  IDs: {results['failed']}")
    print(f"\n  These may need manual download from:")
    for book_id in results['failed']:
        print(f"    https://www.gutenberg.org/ebooks/{book_id}")

In [ ]:
import json
from pathlib import Path

# Paths
log_file = Path('./analysis/reports/nb01-download_log.json')
RAW_DIR = Path('./data/raw')
CLEANED_DIR = Path('./data/processed/cleaned')

# Load the existing log
with open(log_file, 'r', encoding='utf-8') as f:
    full_log = json.load(f)

# --------------------------------------------------------------------
# Count files on disk
# --------------------------------------------------------------------
raw_files = set(f.stem.replace('pg', '') for f in RAW_DIR.glob('pg*.txt'))
cleaned_files = set(f.stem.replace('pg', '') for f in CLEANED_DIR.glob('pg*.txt'))

# Ensure selected_ids is available (it should be from your notebook)
try:
    selected_ids
except NameError:
    # Fallback: use total from the log
    selected_ids = range(full_log['summary']['total_selected'])  # not ideal, but better than nothing
    print("⚠️  'selected_ids' not found – using range based on total_selected.")

# Convert selected_ids to strings for comparison
selected_ids_str = set(str(bid) for bid in selected_ids)

# Compute failures
failed_ids = sorted(selected_ids_str - cleaned_files)

# --------------------------------------------------------------------
# Update the log
# --------------------------------------------------------------------
full_log['download']['successful'] = [{'book_id': bid} for bid in sorted(raw_files)]
full_log['download']['failed'] = [{'book_id': bid, 'error': 'Failed after retry'} for bid in failed_ids]

full_log['cleaning']['successful'] = [{'book_id': bid} for bid in sorted(cleaned_files)]
full_log['cleaning']['failed'] = [{'book_id': bid, 'error': 'No file to clean'} for bid in failed_ids]

full_log['summary'] = {
    'total_selected': len(selected_ids),
    'downloaded': len(raw_files),
    'download_failed': len(failed_ids),
    'cleaned': len(cleaned_files),
    'cleaning_failed': len(failed_ids)
}

# Add retry info if it exists
if 'results' in locals():
    full_log['retry'] = {
        'attempted': len(results['success']) + len(results['failed']),
        'retry_success': len(results['success']),
        'retry_failed': len(results['failed']),
        'special_formats': results.get('special_format', [])
    }

# Save
with open(log_file, 'w', encoding='utf-8') as f:
    json.dump(full_log, f, indent=2, ensure_ascii=False)

print("=" * 60)
print("✅ FINAL CORPUS STATUS (based on files on disk)")
print("=" * 60)
print(f"Total selected:        {full_log['summary']['total_selected']}")
print(f"Downloaded (raw files): {full_log['summary']['downloaded']}")
print(f"Cleaned (final):        {full_log['summary']['cleaned']}")
print(f"Failed (final):         {full_log['summary']['download_failed']}")
print(f"  IDs: {failed_ids[:10]}{'...' if len(failed_ids)>10 else ''}")

# Summary

### Corpus Ready for Distribution

The corpus has been downloaded and cleaned. The `data/` folder contains:

- **`raw/`** - Original texts from Project Gutenberg (with headers/footers)
- **`processed/cleaned/`** - Texts with PG headers/footers removed
- **`/analysis/reports/nb01-download_log.json`** - Detailed log of downloads and cleaning

### Next Steps

1. **Review failed downloads** - Check the log for any missing texts
2. **Distribute corpus** - Share the `corpus/` folder with students (via USB, cloud, etc.)
3. **Notebook 02** - Students will analyze and preprocess these texts

---

## Quick Statistics

In [ ]:
# Quick corpus statistics
import os

total_size_raw = sum(f.stat().st_size for f in RAW_DIR.glob('*.txt'))
total_size_cleaned = sum(f.stat().st_size for f in CLEANED_DIR.glob('*.txt'))

print("Quick corpus statistics")
print("=" * 60)
print(f"Raw texts: {len(list(RAW_DIR.glob('*.txt')))} files")
print(f"Total size (raw): {total_size_raw / (1024*1024):.1f} MB")
print(f"\nCleaned texts: {len(list(CLEANED_DIR.glob('*.txt')))} files")
print(f"Total size (cleaned): {total_size_cleaned / (1024*1024):.1f} MB")
print(f"\nSpace saved by cleaning: {(total_size_raw - total_size_cleaned) / (1024*1024):.1f} MB ({(1 - total_size_cleaned/total_size_raw)*100:.1f}%)")

In [ ]:
# Paths
TEXTS_CLEANED_DIR = Path('./data/processed/cleaned')
OUTPUT_FILE = Path('./analysis/tables/nb01-corpus_statistics.csv')
SUMMARY_FILE = Path('./analysis/reports/nb01-corpus_summary.txt')

# Collect statistics
stats_list = []
total_words = 0
total_chars = 0

print("Calculating corpus statistics...")

for text_file in sorted(TEXTS_CLEANED_DIR.glob('pg*.txt')):
    book_id = text_file.stem.replace('pg', '')
    
    with open(text_file, 'r', encoding='utf-8') as f:
        text = f.read()
    
    # Calculate stats
    char_count = len(text)
    word_count = len(text.split())
    line_count = text.count('\n') + 1
    
    stats_list.append({
        'gutenberg_id': book_id,
        'filename': text_file.name,
        'char_count': char_count,
        'word_count': word_count,
        'line_count': line_count
    })
    
    total_words += word_count
    total_chars += char_count

# Save to CSV
import pandas as pd
df_stats = pd.DataFrame(stats_list)
df_stats.to_csv(OUTPUT_FILE, index=False)

# Create summary report
summary = f"""
CORPUS STATISTICS SUMMARY
{'=' * 60}

Total texts: {len(stats_list)}
Total words: {total_words:,}
Total characters: {total_chars:,}

Average words per text: {total_words / len(stats_list):,.0f}
Median words per text: {df_stats['word_count'].median():,.0f}

Smallest text: {df_stats['word_count'].min():,} words (ID: {df_stats.loc[df_stats['word_count'].idxmin(), 'gutenberg_id']})
Largest text: {df_stats['word_count'].max():,} words (ID: {df_stats.loc[df_stats['word_count'].idxmax(), 'gutenberg_id']})

Corpus size: {total_chars / (1024 * 1024):.2f} MB

Word count quartiles:
  25%: {df_stats['word_count'].quantile(0.25):,.0f} words
  50%: {df_stats['word_count'].quantile(0.50):,.0f} words
  75%: {df_stats['word_count'].quantile(0.75):,.0f} words

{'=' * 60}
Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

with open(SUMMARY_FILE, 'w') as f:
    f.write(summary)

print(summary)
print(f"\n✓ Statistics saved to: {OUTPUT_FILE}")
print(f"✓ Summary saved to: {SUMMARY_FILE}")

```mermaid
flowchart TB
    A0["00<br/>Bootcamp"] --> P1

    subgraph P1["Part I — Corpus building and analysis"]
        direction LR
        A1a["01a<br/>Corpus metadata"] --> A1b["01b<br/>Corpus building"] --> A2["02<br/>Preprocessing"] --> A3["03<br/>Distributions + time"] --> A4a["04a<br/>Lexical exploration"] --> A4b["04b<br/>Embedding"]
    end

    subgraph P2["Part II — Linguistic annotations"]
        direction LR
        A5a["05a<br/>spaCy annotation"] --> A5b["05b<br/>Relation extraction"] --> A6a["06a<br/>NER"] --> A6b["06b<br/>Custom NER"]
    end

    subgraph P3["Part III — Representations"]
        direction LR
        A7["07<br/>BoW + TF-IDF"] --> A8a["08a<br/>Embeddings"] --> A8b["08b<br/>Transformers"]
    end

    subgraph P4["Part IV — Models and interpretation"]
        direction LR
        A9["09<br/>Classification"] --> A10["10<br/>Custom NER training"] --> A11["11<br/>Topic modeling"] --> A12["12<br/>Semantic shift"]
    end

    P1 --> P2
    P2 --> P3
    P3 --> P4

    classDef start fill:#f3f0ff,stroke:#6f42c1,stroke-width:1.5px,color:#111;
    classDef prep fill:#eef7ff,stroke:#1f77b4,stroke-width:1.5px,color:#111;
    classDef annot fill:#eefaf0,stroke:#2ca02c,stroke-width:1.5px,color:#111;
    classDef repr fill:#fff7e6,stroke:#ff8c00,stroke-width:1.5px,color:#111;
    classDef model fill:#fff0f0,stroke:#d62728,stroke-width:1.5px,color:#111;

    classDef highlight fill:#fff3b0,stroke:#f5a623,stroke-width:4px,color:#111;

    class A1a,A1b,A2,A3,A4a,A4b prep;
    class A5a,A5b,A6a,A6b annot;
    class A7,A8a,A8b repr;
    class A9,A10,A11,A12 model;

    class A1b highlight;
```